In [1]:
import io
import zipfile
import requests
import frontmatter

# Downlaod repo

In [2]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [3]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [4]:
repository_data[1]

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

In [5]:
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

In [6]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [7]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1309
Evidently documents: 95


# Chunking

## simple chunking

In [8]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [9]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [10]:
evidently_chunks[0]

{'start': 0,
 'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx'}

## Splitting by Paragraphs and Sections

In [11]:
import re
text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [12]:
paragraphs[4]

"Here's what we'll do:"

In [13]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections

In [14]:
sections = split_markdown_by_level(text, level=2)
sections[0]

'## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.dash

In [15]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

In [16]:
evidently_chunks[0]

{'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx',
 'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>'}

## Intelligent Chunking with LLM

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI

openai_client = OpenAI()


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text


In [4]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [5]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [15]:
evidently_docs[:10]

[{'title': 'Create Plant',
  'openapi': 'POST /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/create.mdx'},
 {'title': 'Delete Plant',
  'openapi': 'DELETE /plants/{id}',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/delete.mdx'},
 {'title': 'Get Plants',
  'openapi': 'GET /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/get.mdx'},
 {'title': 'Introduction',
  'description': 'Example section for showcasing API endpoints',
  'content': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"

In [16]:
from tqdm.auto import tqdm

evidently_chunks = []

for doc in tqdm(evidently_docs[:10]):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

  0%|          | 0/10 [00:00<?, ?it/s]

In [17]:
evidently_chunks

[{'title': 'Create Plant',
  'openapi': 'POST /plants',
  'filename': 'docs-main/api-reference/endpoint/create.mdx',
  'section': 'Certainly! Please provide the content of the document you want to split into sections for the Q&A system.'},
 {'title': 'Delete Plant',
  'openapi': 'DELETE /plants/{id}',
  'filename': 'docs-main/api-reference/endpoint/delete.mdx',
  'section': "Sure! Please provide the document text you'd like me to split into sections."},
 {'title': 'Get Plants',
  'openapi': 'GET /plants',
  'filename': 'docs-main/api-reference/endpoint/get.mdx',
  'section': "It seems that there's no content provided within the document for me to split into sections. Please provide the specific content you'd like me to organize, and I'll be happy to help!"},
 {'title': 'Introduction',
  'description': 'Example section for showcasing API endpoints',
  'filename': 'docs-main/api-reference/introduction.mdx',
  'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenA

# Search

## Lexical search

In [17]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [18]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)

In [19]:
results[:2]

[{'title': 'RAG evaluation dataset',
  'description': 'Synthetic data for RAG.',
  'filename': 'docs-main/synthetic-data/rag_data.mdx',
  'section': '## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthetic_data_inputs_example_upload.png)\n\nSimply drop the file, then:\n\n* Choose the number of inputs to generate.\n* Choose if you want to include the context used to generate the answer.\n\n![](/images/synthetic/synthetic_data_inputs_example_upload2.png)\n\nThe system auto

In [20]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

## Semantic search

In [21]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [22]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)

  0%|          | 0/522 [00:00<?, ?it/s]

In [23]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [24]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)

In [25]:
results[0]

{'id': '3f1424af17',
 'question': 'Course: Can I still join the course after the start date?',
 'sort_order': 3,
 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}

In [26]:
evidently_chunks[0]

{'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx',
 'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>'}

In [27]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    v = embedding_model.encode(d['section'])
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/266 [00:00<?, ?it/s]

## Hybrid search

In [28]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results

In [29]:
hybrid_search(query)

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](htt

# Agents and Tools

In [31]:
import openai
from dotenv import load_dotenv
load_dotenv()

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)

To determine if you can still join a course, you'll need to check a few things:

1. **Enrollment Dates**: See if the enrollment period is still open.
2. **Prerequisites**: Make sure you meet any prerequisites required for the course.
3. **Contacting the Institution**: Reach out to the course provider (like a university or online platform) for specific guidance.

If you let me know which course you're referring to, I might be able to provide more tailored advice!


In [32]:
# build search function
def text_search(query):
    return faq_index.search(query, num_results=5)

In [33]:
# describe the search function
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [34]:
# use search function
system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

In [35]:
# look at output
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course now?"}', call_id='call_18RHzgj5ivgRD59K8UUOfNbF', name='text_search', type='function_call', id='fc_0e2b07f912ff11a80069d7449485648197aca8bddd9e699b25', namespace=None, status='completed')]

In [36]:
# invoke function with arguments from output
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}


In [37]:
# make the LLM stateful
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

Yes, you can still join the course even if you missed the registration deadline! You are eligible to submit homework, but keep in mind that there will be deadlines for submitting assignments and final projects, so it’s best not to procrastinate. 

If you're interested, make sure to stay updated with announcements through the course Telegram channel and set up any required tools beforehand.


In [38]:
# refine system prompt
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""


In [39]:
#write docstring to describe function
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)

In [40]:
# define an agent with PydanticAI
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)


In [41]:
# run the agent
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

In [43]:
result

AgentRunResult(output='Yes, you can join the course now. There’s no need for a confirmation email after registration. You’re accepted into the course automatically and can start learning and submitting homework without any additional registration required. \n\nFeel free to begin exploring the materials and participating! If you have any other questions or need further assistance, just let me know.')

In [47]:
# get detailed information about the agents reasoning and actions. Look at user prompt
result.new_messages()[0]

ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 4, 9, 6, 23, 56, 177463, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 9, 6, 23, 56, 177701, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provide accurate answers.\nIf the search doesn't return relevant results, let the user know and provide general guidance.", run_id='019d70e9-60fc-74ae-b463-aed2a5d7d9ad')

In [48]:
# Take a look at model's response
result.new_messages()[1]

ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"enrollment policy late registration"}', tool_call_id='call_OwqhNvyg1LoTVsSRI3T1qMmI')], usage=RequestUsage(input_tokens=161, output_tokens=18, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 4, 9, 6, 23, 57, 817461, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://api.openai.com/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 4, 9, 6, 23, 57, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-DSd5xtaYHXmx7hlR7bVl7Crnmcv3t', finish_reason='tool_call', run_id='019d70e9-60fc-74ae-b463-aed2a5d7d9ad')

In [49]:
# results returned by the tool
result.new_messages()[2]

ModelRequest(parts=[ToolReturnPart(tool_name='text_search', content=[{'id': '4dbd2eea47', 'question': 'Homework: Are late submissions of homework allowed?', 'sort_order': 17, 'content': 'No, late submissions are not allowed. However, if the form is still open after the due date, you can still submit the homework. Confirm your submission by checking the date-timestamp on the Course page. Ensure you are logged in.', 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/017_4dbd2eea47_homework-are-late-submissions-of-homework-allowed.md'}, {'id': '06c61761df', 'question': "GCP: How do I create a service account key when 'service account key creation is disabled'?", 'sort_order': 138, 'content': 'This error occurs when your organization has a policy that disables service account key creation. You can disable this policy using either the GCP Console or gcloud CLI.\n\n### Option 1: Using GCP Console\n\n1. Go to the [IAM dashboard](https://console.cloud.google.com/iam-admin/iam) 

In [50]:
# model response
result.new_messages()[3]

ModelResponse(parts=[TextPart(content='Yes, you can join the course now. There’s no need for a confirmation email after registration. You’re accepted into the course automatically and can start learning and submitting homework without any additional registration required. \n\nFeel free to begin exploring the materials and participating! If you have any other questions or need further assistance, just let me know.')], usage=RequestUsage(input_tokens=1443, output_tokens=68, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 4, 9, 6, 24, 0, 139461, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://api.openai.com/v1/', provider_details={'finish_reason': 'stop', 'timestamp': datetime.datetime(2026, 4, 9, 6, 23, 59, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-DSd5ztLTfzsW2FXxPEAb228yxfchM', finish_reason='stop', run_id='0